## Decision Tree Classifier

A decision tree is a flowchart-like tree structure where an internal node represents a feature, the branch represents a decision rule, and each leaf node represents the outcome.

You do not need to standardize or normalize data for decision tree models. Decision trees are scale-invariant, meaning they make decisions based on the order of values rather than their absolute magnitude.

Tutorial guide: https://www.datacamp.com/tutorial/decision-tree-classification-python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn import metrics
import seaborn as sns

RANDOM_SEED = 42

Read data

In [ ]:
train_df = pd.read_csv('../../data/processed/tree/train_data.csv')
val_df = pd.read_csv('../../data/processed/tree/val_data.csv')
test_df = pd.read_csv('../../data/processed/tree/test_data.csv')

In [ ]:
X_train = train_df.drop('fraud', axis=1)
y_train = train_df['fraud']

X_val = val_df.drop('fraud', axis=1)
y_val = val_df['fraud']

X_test = test_df.drop('fraud', axis=1)
y_test = test_df['fraud']

In [ ]:
param_grid = [
    {"max_depth": 3, "min_samples_leaf": 1, "min_samples_split": 2},
    {"max_depth": 5, "min_samples_leaf": 1, "min_samples_split": 2},
    {"max_depth": 8, "min_samples_leaf": 1, "min_samples_split": 2},
    {"max_depth": 8, "min_samples_leaf": 5, "min_samples_split": 10},
    {"max_depth": 10, "min_samples_leaf": 10, "min_samples_split": 20},
]

Build decision tree model

In [ ]:
results = []

for params in param_grid:
    clf = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_SEED, criterion="gini", **params)
    clf.fit(X_train, y_train)
    y_val_pred = clf.predict(X_val)
    
    results.append({
        **params,
        "accuracy": metrics.accuracy_score(y_val, y_val_pred),
        "precision": metrics.precision_score(y_val, y_val_pred, zero_division=0),
        "recall": metrics.recall_score(y_val, y_val_pred, zero_division=0),
        "f1": metrics.f1_score(y_val, y_val_pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)
print(results_df)

In [ ]:
best_params = results_df.iloc[0][["max_depth", "min_samples_leaf", "min_samples_split"]].to_dict()

train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df.drop('fraud', axis=1)
y_train_val = train_val_df['fraud']

clf_final = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=RANDOM_SEED,
    criterion="gini",
    max_depth=int(best_params["max_depth"]),
    min_samples_leaf=int(best_params["min_samples_leaf"]),
    min_samples_split=int(best_params["min_samples_split"])
)

clf_final.fit(X_train_val, y_train_val)
y_test_pred = clf_final.predict(X_test)

accuracy = metrics.accuracy_score(y_test, y_test_pred)
precision = metrics.precision_score(y_test, y_test_pred, zero_division=0)
recall = metrics.recall_score(y_test, y_test_pred, zero_division=0)
f1 = metrics.f1_score(y_test, y_test_pred, zero_division=0)

print("Final test accuracy:", accuracy)
print("Final test precision:", precision)
print("Final test recall:", recall)
print("Final test f1:", f1)

In [ ]:
plt.figure(figsize=(40,20))
plot_tree(clf, filled=True)
plt.tight_layout()
plt.title('Decision Tree Classifier', fontsize=20)
plt.figtext(0.03, 0.03, f'Accuracy: {accuracy}\nPrecision: {precision}\nRecall: {recall}\nF1 Score: {f1}', ha='left', fontsize=15)
plt.savefig("../../results/DecisionTree.svg")
plt.show()

In [ ]:
cf_matrix = metrics.confusion_matrix(y_test, y_test_pred)

group_names = ["True Negatives", "False Positives", "False Negatives", "True Positives"]

group_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_names,group_counts)]
labels = np.array(labels).reshape(2,2)

matrix =sns.heatmap(metrics.confusion_matrix(y_test, y_test_pred), annot=labels, fmt='', cmap="viridis", cbar=False)
matrix.set_xlabel("Predicted Label")
matrix.set_ylabel("True Label")
matrix.set_xticklabels(["Negative", "Positive"])
matrix.set_yticklabels(["Negative", "Positive"])
plt.title('Confusion Matrix')
plt.savefig("../../results/DecisionTreeCM.svg")
plt.show()

In [ ]:
feat_imp = pd.Series(clf.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=True)[:15]

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh')
plt.title("Top 15 Feature Importance (Decision Tree)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    clf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

perm_imp = pd.Series(result.importances_mean, index=X_train.columns)
perm_imp = perm_imp.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
perm_imp.sort_values().plot(kind='barh')
plt.title("Permutation Feature Importances")
plt.xlabel("Decrease in Model Performance")
plt.tight_layout()
plt.show()